# Phase 8.1 — K-Means: Crime Hotspot Clustering

**Goal:** group crime locations into geographic zones ("hotspots") using latitude and longitude only.

**Input:** `data/processed/coordinates.parquet` — 150,000 randomly sampled incidents with valid coordinates (seed 42).
**Script:** `ml/train_kmeans.py` (this notebook reads its saved results; it does not retrain unless you run the last cell).

## 1. Method

- **Features:** latitude and longitude only (Master Plan, Phase 8.1).
- **Projection to kilometres:** one degree of longitude is shorter than one degree of latitude at Chicago's latitude (about 83 km vs 111 km). Clustering raw degrees would stretch east–west distances, so coordinates are converted to kilometres first. The conversion constants are saved inside `kmeans.pkl` so the API applies the same transform.
- **Choosing K:** K = 5 to 15 was tried. For each K we record *inertia* (elbow method) and the *silhouette score* (computed on a 10,000-point sample for speed).
- **Risk tier:** clusters are ranked by size (share of recorded incidents) and split into High / Medium / Low thirds. This is a **relative incident-volume tier**, not a population-adjusted risk score.

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

BASE = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
RESULTS, MODELS, SHOTS = f"{BASE}/ml/results", f"{BASE}/ml/models", f"{BASE}/docs/screenshots"
pd.set_option("display.width", 200)

## 2. Input data

In [ ]:
coords = pd.read_parquet(f"{BASE}/data/processed/coordinates.parquet")
print("Rows:", len(coords), "| unique coordinate pairs:", coords[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Missing lat/long:", int(coords[["latitude", "longitude"]].isna().sum().sum()))
display(coords[["latitude", "longitude"]].agg(["min", "max"]))

## 3. Choosing K

In [ ]:
scores = pd.read_csv(f"{RESULTS}/kmeans_k_scores.csv")
display(scores.round(4))

# column names are detected so this cell works with either naming
k_col = next(c for c in scores.columns if c.lower() == "k" or "cluster" in c.lower())
in_col = next(c for c in scores.columns if "inertia" in c.lower())
si_col = next(c for c in scores.columns if "silhouette" in c.lower())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(scores[k_col], scores[in_col], marker="o"); ax[0].set_title("Elbow method"); ax[0].set_xlabel("K"); ax[0].set_ylabel("Inertia")
ax[1].plot(scores[k_col], scores[si_col], marker="o", color="#c0392b"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("K"); ax[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()
best = scores.loc[scores[si_col].idxmax()]
print(f"Highest silhouette: K={int(best[k_col])} ({best[si_col]:.4f})")
print(scores.set_index(k_col)[si_col].loc[[7, 8, 10]].round(4).to_dict())

## 4. Final model: K = 8

In [ ]:
clusters = pd.read_csv(f"{RESULTS}/hotspot_clusters.csv")
display(clusters.drop(columns=[c for c in ["risk_basis"] if c in clusters.columns]).round(3))
print(clusters["risk_level"].value_counts().to_dict())

In [ ]:
display(Image(f"{SHOTS}/11_kmeans_clusters.png"))

## 5. Using the saved model (what the API does)

In [ ]:
store = joblib.load(f"{MODELS}/kmeans.pkl")
print("Stored keys:", list(store.keys()))
print("K =", store["k"], "| features:", store["features"])
if "sampled_silhouette_score" in store:
    print("Sampled silhouette:", round(store["sampled_silhouette_score"], 4))

def assign_cluster(lat, lon):
    x = np.array([[lat * store["km_per_deg_lat"], lon * store["km_per_deg_lon"]]])
    c = int(store["model"].predict(x)[0])
    return c, store["risk_levels"][c]

# Example: downtown Chicago
c, tier = assign_cluster(41.8781, -87.6298)
print(f"Point (41.8781, -87.6298) -> cluster {c}, relative incident-volume tier: {tier}")
print(clusters.loc[clusters['cluster'] == c, ['cluster', 'size', 'share_pct', 'top_crime_type', 'risk_level']].to_string(index=False))

## 6. Interpretation and limitations

- **Why K = 8:** silhouette scores are almost flat for K = 7 to 10 (K=7: 0.4114, K=8: 0.4103, K=10: 0.4103), and the elbow curve has no sharp bend. K = 7 scores 0.0011 higher, which is not a meaningful difference. K = 8 was chosen because it gives a clearer geographic breakdown of the city.
- **What the clusters are:** K-Means on coordinates splits the city into compact regions, so these are better described as *activity zones* than as small dense hotspots. Crime is recorded across the whole city, so every area ends up in some cluster.
- **Top crime type per cluster:** THEFT leads in most central and northern clusters, BATTERY in southern and western ones, and "ALL OTHER" (a catch-all category) leads in one cluster. Each cluster's top crime group accounts for only about 19%–36% of its incidents, so the mix is varied.
- **Risk tiers:** High / Medium / Low show relative incident volume only. They are not adjusted for population or for how many people visit an area, so they should not be read as the danger to a resident.
- **Limitations:** K-Means assumes round clusters of similar size, cannot mark noise points, and uses a 150,000-incident sample (about 16% of all records).

## 7. (Optional) Re-run the experiment

In [ ]:
# Retrains 11 models on 150,000 points; takes a few minutes. Uncomment to run.
# %run ../ml/train_kmeans.py 8